In [11]:
import langchain



In [9]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [21]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [7]:
### example 1: Simple LLM call with streaming

In [9]:
from langchain.chat_models import init_chat_model
#for creating the chat model = init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

In [11]:
model = init_chat_model("groq:openai/gpt-oss-20b")
model


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000239615FAF90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000239615FBA10>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [12]:
from langchain_groq import ChatGroq
llm=ChatGroq(model="openai/gpt-oss-20b")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001D1964101A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D196410C20>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [16]:
## create messages
messages=[
    SystemMessage("You are a helpful AI assistant"),
    HumanMessage("What are the top 2 benefits of using LANGCHAIN")
]

In [17]:
response = model.invoke(messages)
response

AIMessage(content='**Top 2 Benefits of Using LangChain**\n\n| # | Benefit | Why It Matters |\n|---|---------|----------------|\n| **1. Modular, reusable components** | LangChain breaks an LLM‑powered application into small, composable pieces (chains, prompts, memory, tools, etc.). This lets you swap, stack, or extend functionality without rewriting the whole system. |\n| **2. Rapid prototyping & production‑grade tooling** | With pre‑built connectors for APIs, databases, and vector stores, you can spin up a working prototype in minutes and then scale it with confidence—thanks to built‑in logging, error handling, and performance monitoring. |\n\nThese two strengths make LangChain a go‑to choice for quickly building, testing, and deploying robust LLM applications.', additional_kwargs={'reasoning_content': 'We need to answer: "What are the top 2 benefits of using LANGCHAIN". The user is presumably asking about LangChain, a framework for building LLM applications. We need to provide top 2 b

In [18]:
print(response.content)

**Top 2 Benefits of Using LangChain**

| # | Benefit | Why It Matters |
|---|---------|----------------|
| **1. Modular, reusable components** | LangChain breaks an LLM‑powered application into small, composable pieces (chains, prompts, memory, tools, etc.). This lets you swap, stack, or extend functionality without rewriting the whole system. |
| **2. Rapid prototyping & production‑grade tooling** | With pre‑built connectors for APIs, databases, and vector stores, you can spin up a working prototype in minutes and then scale it with confidence—thanks to built‑in logging, error handling, and performance monitoring. |

These two strengths make LangChain a go‑to choice for quickly building, testing, and deploying robust LLM applications.


In [20]:
##streaming example
for chunk in model.stream(messages):
    print(chunk.content, end="", flush=True)

**Top 2 Benefits of Using LangChain**

1. **Modular, Plug‑and‑Play Architecture**  
   - LangChain breaks down the LLM workflow into reusable components (prompt templates, memory stores, output parsers, chain orchestration, etc.).  
   - This lets you quickly assemble or swap out pieces—e.g., replace a retrieval module or add a new prompt strategy—without rewriting your entire pipeline.

2. **Rich Ecosystem & Community‑Driven Extensibility**  
   - A large library of pre‑built connectors (databases, APIs, web scrapers, vector stores) and integrations (OpenAI, Anthropic, Azure, etc.) accelerates development.  
   - Community contributions (plugins, templates, and best‑practice guides) keep the framework up‑to‑date and enable you to adopt cutting‑edge techniques with minimal effort.

In [21]:
## Dynamic prompt template


In [25]:
from langchain_core.prompts import ChatPromptTemplate

##create translation app

translation_template= ChatPromptTemplate.from_messages([
    ("system", "You are a professional translator. Translate the following text-{text} to {source_language} to {target_language}. Maintain the tone and style"),
    ("user", "{text}")
])

##using the template
prompt=translation_template.invoke({
    "source_language":"English",
    "target_language":"Spanish",
    "text":"I love India and India has so many cultures, cuisines"
})

In [26]:
prompt

ChatPromptValue(messages=[SystemMessage(content='You are a professional translator. Translate the following text-I love India and India has so many cultures, cuisines to English to Spanish. Maintain the tone and style', additional_kwargs={}, response_metadata={}), HumanMessage(content='I love India and India has so many cultures, cuisines', additional_kwargs={}, response_metadata={})])

In [27]:
translated_response=model.invoke(prompt)
print(translated_response.content)

Me encanta India y tiene tantas culturas y cocinas.


In [28]:
##BUILDING Your First Chain


In [48]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()


model=ChatGroq(model="openai/gpt-oss-20b")

In [49]:

def create_story_chain():
    #prompt for story generation
    story_prompt = ChatPromptTemplate.from_messages([
        ("system","You are a creative storyteller. Write a short and engaging story based on a given theme"),
        ("user","Theme:{theme}\n Main character:{character} \n Setting:{setting}")
    ])

  ##template for story analysis
    analysis_prompt=ChatPromptTemplate.from_messages([
        ("system","You are a literally critic. Analyze the following story and provide insights"),
        ("user","{story}")
    ])

    story_chain=(
      story_prompt | model | StrOutputParser()
    )

#create a function to pass the story to analysis
    def analyze_story(story_text):
       return{"story": story_text}

    analysis_chain = (
      story_chain
      | RunnableLambda(analyze_story)
      | analysis_prompt
      | model
      | StrOutputParser()
    )

    return analysis_chain

In [50]:
chain = create_story_chain()
chain

ChatPromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a creative storyteller. Write a short and engaging story based on a given theme'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, template='Theme:{theme}\n Main character:{character} \n Setting:{setting}'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': 

In [51]:
print(chain)

first=ChatPromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a creative storyteller. Write a short and engaging story based on a given theme'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, template='Theme:{theme}\n Main character:{character} \n Setting:{setting}'), additional_kwargs={})]) middle=[ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audi

In [52]:
result = chain.invoke({
    "theme": "AI",
    "character": "a curious person",
    "setting": "a futuristic city"
})

print("Story and Analysis:")  #simply to display above in the output, nothing imp
print(result)

Story and Analysis:
**Title (unnamed)**  
**Genre:** Cyber‑punk / speculative fiction  
**Tone:** Lyrical, almost poetic, with a touch of wistfulness

---

## 1. Premise & World‑Building

| Strength | Weakness |
|----------|----------|
| **Vivid, sensory cityscape** – Neo‑Solis is introduced as a “neon‑lit arteries” city with sky‑bridges, drones, and a central AI (Helix). The imagery is evocative and instantly places the reader in a high‑tech metropolis. | **Exposition overload** – The city’s description feels a bit like a list of tropes (neon, sky‑bridges, drones). While evocative, it risks feeling generic to seasoned cyber‑punk readers. |
| **Historical depth** – The Data Tower as a relic of the pre‑AI era and the Echo as a “remnant of Helix’s early architecture” gives the world a sense of evolution and a hidden past. | **Lack of concrete stakes** – The story hinges on the idea that Helix “forgot” human memories, but the stakes of this forgetting (social, political, emotional) are no